<a href="https://colab.research.google.com/github/SitthisakMoukomla/ricebiomass/blob/main/ricebiomass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import ee
import geemap
import pandas as pd

# === Authenticate and Initialize ===
ee.Authenticate()
ee.Initialize(project='ee-pythoncolab')

# === Define AOI: จังหวัดนครสวรรค์ ===
province = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(
    ee.Filter.eq('ADM1_NAME', 'Nakhon Sawan'))
aoi = province.geometry()

# === Load Sentinel-2 and calculate GCVI ===
def add_gcvi(img):
    gcvi = img.expression(
        'NIR / GREEN - 1',
        {'NIR': img.select('B8'), 'GREEN': img.select('B3')}
    ).rename('GCVI')
    return img.addBands(gcvi)

s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED') \
    .filterBounds(aoi) \
    .filterDate('2024-11-01', '2024-12-15') \
    .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20)) \
    .map(add_gcvi)

gcvi_mean = s2.select('GCVI').median().clip(aoi)

# === Threshold GCVI < 1.0 → เก็บเกี่ยวแล้ว ===
harvested_area = gcvi_mean.lt(1.0).selfMask()

# === คำนวณพื้นที่ (m² → ไร่) ===
pixelArea = ee.Image.pixelArea().updateMask(harvested_area)
area_stats = pixelArea.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=aoi,
    scale=10,
    maxPixels=1e10
)
area_m2 = area_stats.getNumber('area')
area_rai = area_m2.divide(1600)

# === คำนวณตันฟาง และรถเกี่ยว ===
rai = area_rai.getInfo()
tons = round(rai * 0.18, 2)
trips = int(tons / 4)

print(f"📍 พื้นที่เกี่ยวแล้ว ≈ {rai:.0f} ไร่")
print(f"📦 ปริมาณฟางข้าว ≈ {tons} ตัน")
print(f"🚜 รถเกี่ยวที่ต้องใช้ ≈ {trips} คัน (เที่ยวละ 4 ตัน)")


📍 พื้นที่เกี่ยวแล้ว ≈ 979967 ไร่
📦 ปริมาณฟางข้าว ≈ 176394.06 ตัน
🚜 รถเกี่ยวที่ต้องใช้ ≈ 44098 คัน (เที่ยวละ 4 ตัน)


In [5]:
Map = geemap.Map(center=[15.7, 100.1], zoom=11)
Map.addLayer(gcvi_mean, {"min": 0, "max": 2, "palette": ['yellow', 'green', 'darkgreen']}, "GCVI")
Map.addLayer(harvested_area, {"palette": ['pink']}, "พื้นที่เกี่ยวแล้ว")
Map


Map(center=[15.7, 100.1], controls=(WidgetControl(options=['position', 'transparent_bg'], widget=SearchDataGUI…

In [7]:
pip install streamlit geemap earthengine-api pandas


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 81.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 93.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 53.6 MB/s eta 0:00:00


In [11]:
import streamlit as st
import ee
import geemap.foliumap as geemap

# Init EE
ee.Authenticate()
ee.Initialize(project='ee-pythoncolab')

# Header
st.set_page_config(layout="wide")
st.title("ระบบติดตาม GCVI พื้นที่เกี่ยวข้าว")
st.subheader("กล้า-แกร่ง • จังหวัดนครสวรรค์")

# ดึงข้อมูล GCVI
def get_gcvi():
    province = ee.FeatureCollection("FAO/GAUL/2015/level1").filter(ee.Filter.eq('ADM1_NAME', 'Nakhon Sawan'))
    aoi = province.geometry()
    def add_gcvi(img):
        gcvi = img.expression('NIR / GREEN - 1', {
            'NIR': img.select('B8'),
            'GREEN': img.select('B3')
        }).rename('GCVI')
        return img.addBands(gcvi)
    s2 = ee.ImageCollection('COPERNICUS/S2_HARMONIZED')\
        .filterBounds(aoi)\
        .filterDate('2024-11-01', '2024-12-15')\
        .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 20))\
        .map(add_gcvi)
    return s2.select('GCVI').median().clip(aoi)

# ประมวลผล
gcvi_image = get_gcvi()
gcvi_stats = gcvi_image.lt(1.0).selfMask()
pixel_area = ee.Image.pixelArea().updateMask(gcvi_stats)
# Increase maxPixels and set bestEffort to True
area_rai = pixel_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=gcvi_stats.geometry(),
    scale=10,
    maxPixels=1e13, # Increased maxPixels to 1e13
    bestEffort=True # Added bestEffort to handle large computations
).getNumber('area').divide(1600)
tons = area_rai.multiply(0.18)
trips = tons.divide(4)

# แสดงผล
st.metric("พื้นที่เกี่ยวแล้ว (ไร่)", f"{area_rai.getInfo():,.0f}")
st.metric("ปริมาณฟางโดยประมาณ (ตัน)", f"{tons.getInfo():,.0f}")
st.metric("รถเกี่ยวที่ต้องใช้ (เที่ยว)", f"{trips.getInfo():,.0f}")

# แสดงแผนที่
m = geemap.Map()
m.centerObject(gcvi_image, 8)
m.addLayer(gcvi_image, {"min": 0, "max": 2, "palette": ['yellow', 'green', 'darkgreen']}, "GCVI")
m.addLayer(gcvi_stats, {"palette": ['red']}, "เกี่ยวแล้ว")
m.to_streamlit(height=600)

2025-05-02 16:29:09.486 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 16:29:09.489 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 16:29:09.491 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 16:29:09.493 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 16:29:09.494 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 16:29:56.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 16:29:56.007 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-05-02 16:29:56.997 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

DeltaGenerator()